# 06_mllm / gemini_setup.ipynb

**Purpose:** One-time setup for the Gemini 2.5 Flash multimodal LLM used to generate human-readable explanations of GSAF predictions.

**What this notebook does:**
1. Installs `google-generativeai` if missing.
2. Loads your Gemini API key from `.env` or prompts you to paste it.
3. Verifies the API key with a text-only call.
4. Verifies vision works with a single image-caption pair from the dataset.
5. Saves a small helper module (`src/utils/gemini_client.py`) that the next two notebooks (`mllm_inference.ipynb`, `chain_of_thought.ipynb`) will import.
6. Sets up a disk-backed cache so repeat calls do not re-bill the API.

**Free-tier reminder (Gemini 2.5 Flash, as of April 2026):**
- ~10 requests / minute
- ~250 requests / day
- The next two notebooks will run on a stratified 500-pair sample; that takes ~2 calendar days at free-tier limits.

**Inputs:**
- A Gemini API key from https://aistudio.google.com/apikey
- Stored in `.env` at repo root as `GEMINI_API_KEY=...`
- Optional: one local image from `data/processed/ooc_pairs/images/` for the smoke test

**Outputs:**
- `src/utils/gemini_client.py` — the shared client + cache module
- `cache/mllm/.gemini_setup_ok` — sentinel file so future notebooks can verify setup ran


In [1]:
# 1. Install dependency (idempotent)
import subprocess, sys
try:
    import google.generativeai as genai
    print("google-generativeai already installed:", genai.__version__ if hasattr(genai, "__version__") else "ok")
except ImportError:
    print("Installing google-generativeai…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-generativeai", "python-dotenv", "Pillow"])
    import google.generativeai as genai
    print("Installed.")


Installing google-generativeai…
Installed.


C:\Users\User\AppData\Local\Temp\ipykernel_12336\172896496.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
# 2. Path setup
from __future__ import annotations
import os, sys, json, hashlib, time
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent
        break
sys.path.insert(0, str(ROOT))

from src.utils.paths import ALL_MANIFEST, PROCESSED_IMAGES, VERITE_IMAGES

CACHE_MLLM = ROOT / "cache" / "mllm"
CACHE_MLLM.mkdir(parents=True, exist_ok=True)
GEMINI_CACHE = CACHE_MLLM / "gemini_responses"
GEMINI_CACHE.mkdir(parents=True, exist_ok=True)

ENV_PATH = ROOT / ".env"
print("Repo root:", ROOT)
print("MLLM cache:", CACHE_MLLM)
print(".env exists:", ENV_PATH.exists())


Repo root: e:\eve-rag
MLLM cache: e:\eve-rag\cache\mllm
.env exists: True


## 3. Load API key

If you don't have one yet:
1. Visit https://aistudio.google.com/apikey
2. Click "Create API key"
3. Add this line to `E:\\eve-rag\\.env`:
   ```
   GEMINI_API_KEY=AIzaSy…
   ```
4. Re-run this cell.


In [3]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=ENV_PATH, override=True)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
if not GEMINI_API_KEY:
    GEMINI_API_KEY = input("Paste your GEMINI_API_KEY (won't be saved): ").strip()

assert GEMINI_API_KEY, "No API key provided. Add GEMINI_API_KEY to .env or paste when prompted."
assert GEMINI_API_KEY.startswith("AIza") or len(GEMINI_API_KEY) > 20, "API key looks malformed."
print(f"API key loaded (length={len(GEMINI_API_KEY)}, prefix={GEMINI_API_KEY[:6]}…)")


API key loaded (length=39, prefix=AIzaSy…)


## 4. Configure the SDK and pick a model

In [4]:
import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)

# List available models that support vision
print("Listing vision-capable models:")
vision_models = []
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        if "vision" in m.name.lower() or "flash" in m.name.lower() or "pro" in m.name.lower():
            print(f"  {m.name}")
            vision_models.append(m.name)
print(f"\nFound {len(vision_models)} candidate models.")


Listing vision-capable models:
  models/gemini-2.5-flash
  models/gemini-2.5-pro
  models/gemini-2.0-flash
  models/gemini-2.0-flash-001
  models/gemini-2.0-flash-lite-001
  models/gemini-2.0-flash-lite
  models/gemini-2.5-flash-preview-tts
  models/gemini-2.5-pro-preview-tts
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-pro-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-image
  models/gemini-3-pro-preview
  models/gemini-3-flash-preview
  models/gemini-3.1-pro-preview
  models/gemini-3.1-pro-preview-customtools
  models/gemini-3.1-flash-lite-preview
  models/gemini-3-pro-image-preview
  models/nano-banana-pro-preview
  models/gemini-3.1-flash-image-preview
  models/lyria-3-pro-preview
  models/gemini-3.1-flash-tts-preview
  models/deep-research-pro-preview-12-2025

Found 24 candidate models.


In [5]:
# Pick the model we want — Gemini 2.5 Flash is the recommended free-tier default
PREFERRED = [
    "models/gemini-2.5-flash",
    "models/gemini-2.0-flash",
    "models/gemini-1.5-flash",
    "models/gemini-1.5-flash-latest",
]
MODEL_NAME = None
for cand in PREFERRED:
    if cand in vision_models:
        MODEL_NAME = cand
        break
if MODEL_NAME is None and vision_models:
    MODEL_NAME = vision_models[0]
assert MODEL_NAME is not None, "No usable Gemini model found in your account."
print("Using model:", MODEL_NAME)


Using model: models/gemini-2.5-flash


## 5. Smoke test 1 — text only

In [6]:
model = genai.GenerativeModel(MODEL_NAME)
resp = model.generate_content("Reply with exactly: 'EVE-RAG setup OK'")
print("Text reply:", resp.text.strip())
assert "EVE-RAG" in resp.text or "OK" in resp.text, "Unexpected text reply"


Text reply: EVE-RAG setup OK


## 6. Smoke test 2 — image + caption

In [7]:
import pandas as pd
from PIL import Image

# Find one row with a real local image (prefer mmfakebench, then verite)
m = pd.read_csv(ALL_MANIFEST)
candidates = m[(m["has_image"] == 1) & (m["dataset"].isin(["mmfakebench", "verite"]))].head(50)

test_row = None
test_img = None
for _, r in candidates.iterrows():
    p = Path(str(r.get("image_path") or ""))
    if not p.is_absolute():
        for base in [PROCESSED_IMAGES, VERITE_IMAGES, ROOT]:
            cand = base / p.name
            if cand.exists():
                p = cand
                break
            cand2 = base / p
            if cand2.exists():
                p = cand2
                break
    if p.exists():
        test_row = r
        test_img = p
        break

assert test_row is not None, "Could not find a local image for smoke test"
print("Test image:", test_img)
print("Caption   :", test_row["caption"][:140])
print("Label     :", test_row["label"])

img = Image.open(test_img).convert("RGB")
img.thumbnail((512, 512))

prompt = (
    "You are an analyst checking whether an image and a caption match. "
    "In one sentence, describe what is shown in the image. "
    "In one more sentence, say whether the image plausibly matches the caption: "
    f"'{test_row['caption']}'."
)

t0 = time.time()
resp = model.generate_content([prompt, img])
dt = time.time() - t0
print(f"\nReply ({dt:.1f}s):")
print(resp.text.strip())


C:\Users\User\AppData\Local\Temp\ipykernel_12336\1778284351.py:5: DtypeWarning: Columns (0: image_path, 1: source_image_id, 2: text_source, 3: image_source, 4: fake_cls, 5: raw_label) have mixed types. Specify dtype option on import or set low_memory=False.
  m = pd.read_csv(ALL_MANIFEST)


Test image: E:\eve-rag\data\processed\ooc_pairs\images\verite_true_0.jpg
Caption   : Photograph shows a family that was taken away by local epidemic prevention personnel after they had returned home from Hubei Province in Feb
Label     : 0

Reply (8.1s):
The image shows multiple individuals, some in full white hazmat suits, forcefully removing at least one person from what appears to be a residential apartment in a hallway.

The image plausibly matches the caption, as it depicts epidemic prevention personnel in hazmat suits involved in the forced removal of individuals from a home, consistent with strict COVID-19 quarantine enforcement measures that occurred in China, particularly around Hubei Province, in February 2020.


## 7. Build the shared helper module

In [8]:
HELPER_PATH = ROOT / "src" / "utils" / "gemini_client.py"

helper_code = '''"""
Shared Gemini 2.5 Flash client for EVE-RAG.

Usage from a notebook:
    from src.utils.gemini_client import GeminiClient
    gc = GeminiClient()
    text = gc.generate(prompt, image_path=Path("x.jpg"))

Features:
- Disk-backed cache (`cache/mllm/gemini_responses/`) keyed by hash(prompt + image_bytes + model).
- Soft rate limit: at most N_RPM requests per minute (default 10).
- Retries on transient errors (429, 500, network).
- Returns plain text; raises only on fatal config / auth failures.
"""
from __future__ import annotations
import os, json, time, hashlib
from pathlib import Path
from dataclasses import dataclass

import google.generativeai as genai
from PIL import Image
from dotenv import load_dotenv


@dataclass
class GeminiClient:
    model_name: str = "MODEL_NAME_PLACEHOLDER"
    rpm_limit: int = 10
    cache_dir: Path = None
    image_max_side: int = 512
    max_retries: int = 4
    request_log: list = None

    def __post_init__(self):
        # Locate repo root
        here = Path(__file__).resolve()
        for p in [here] + list(here.parents):
            if (p / "src" / "utils" / "paths.py").exists():
                self.repo_root = p
                break
        else:
            self.repo_root = Path.cwd()
        # API key
        load_dotenv(self.repo_root / ".env", override=True)
        api_key = os.environ.get("GEMINI_API_KEY", "").strip()
        if not api_key:
            raise RuntimeError("GEMINI_API_KEY not in environment. Put it in .env at repo root.")
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(self.model_name)
        # Cache
        if self.cache_dir is None:
            self.cache_dir = self.repo_root / "cache" / "mllm" / "gemini_responses"
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        # Rate limit state
        self.request_log = []

    # --- cache key ---
    @staticmethod
    def _key(prompt: str, image_path: Path | None, model_name: str) -> str:
        h = hashlib.sha256()
        h.update(model_name.encode())
        h.update(b"||")
        h.update(prompt.encode("utf-8"))
        h.update(b"||")
        if image_path is not None and Path(image_path).exists():
            h.update(Path(image_path).read_bytes())
        return h.hexdigest()[:32]

    # --- soft rate limit ---
    def _wait_if_needed(self):
        now = time.time()
        self.request_log = [t for t in self.request_log if now - t < 60]
        if len(self.request_log) >= self.rpm_limit:
            sleep_for = 60 - (now - self.request_log[0]) + 0.1
            if sleep_for > 0:
                time.sleep(sleep_for)
                self.request_log = [t for t in self.request_log if time.time() - t < 60]

    def generate(self, prompt: str, image_path: Path | None = None,
                 use_cache: bool = True, temperature: float = 0.2,
                 max_output_tokens: int = 512) -> str:
        """Return Gemini's text response. Caches by hash of (prompt, image, model)."""
        key = self._key(prompt, image_path, self.model_name)
        cache_file = self.cache_dir / f"{key}.json"

        if use_cache and cache_file.exists():
            try:
                d = json.loads(cache_file.read_text(encoding="utf-8"))
                return d["text"]
            except Exception:
                pass

        self._wait_if_needed()
        parts: list = [prompt]
        if image_path is not None:
            img = Image.open(image_path).convert("RGB")
            img.thumbnail((self.image_max_side, self.image_max_side))
            parts.append(img)

        gen_cfg = {"temperature": temperature, "max_output_tokens": max_output_tokens}
        last_err = None
        for attempt in range(self.max_retries):
            try:
                self.request_log.append(time.time())
                resp = self.model.generate_content(parts, generation_config=gen_cfg)
                text = (resp.text or "").strip()
                cache_file.write_text(json.dumps({
                    "text": text, "prompt": prompt[:2000],
                    "image": str(image_path) if image_path else None,
                    "model": self.model_name, "ts": time.time(),
                }), encoding="utf-8")
                return text
            except Exception as e:
                last_err = e
                msg = str(e).lower()
                if any(k in msg for k in ("429", "quota", "rate", "deadline", "503", "500", "unavailable")):
                    backoff = min(60.0, 2 ** attempt * 5)
                    time.sleep(backoff)
                    continue
                raise
        raise RuntimeError(f"Gemini call failed after {self.max_retries} retries: {last_err}")
'''

helper_code = helper_code.replace("MODEL_NAME_PLACEHOLDER", MODEL_NAME)
HELPER_PATH.write_text(helper_code, encoding="utf-8")
print("Wrote helper module:", HELPER_PATH)
print(f"({len(helper_code):,} chars)")


Wrote helper module: e:\eve-rag\src\utils\gemini_client.py
(4,715 chars)


## 8. Verify helper module works end-to-end

In [9]:
# Reimport with the new helper
import importlib
if "src.utils.gemini_client" in sys.modules:
    importlib.reload(sys.modules["src.utils.gemini_client"])
from src.utils.gemini_client import GeminiClient

gc = GeminiClient()
print("Helper loaded. Model =", gc.model_name)

# Cached call (should be <0.1s if smoke-test response was cached, else a fresh call)
out = gc.generate(
    prompt="In one sentence, what does this image show?",
    image_path=test_img,
    use_cache=True,
    temperature=0.0,
    max_output_tokens=120,
)
print("\nHelper test reply:")
print(out)


Helper loaded. Model = models/gemini-2.5-flash

Helper test reply:
The image shows a


## 9. Write sentinel file

In [10]:
sentinel = CACHE_MLLM / ".gemini_setup_ok"
sentinel.write_text(json.dumps({
    "model_name": MODEL_NAME,
    "ts": time.time(),
    "rpm_limit": 10,
}, indent=2), encoding="utf-8")
print("Wrote sentinel:", sentinel)
print()
print("✓ Setup complete. Proceed to 06_mllm/mllm_inference.ipynb")


Wrote sentinel: e:\eve-rag\cache\mllm\.gemini_setup_ok

✓ Setup complete. Proceed to 06_mllm/mllm_inference.ipynb


---
**Done.** The next notebook (`mllm_inference.ipynb`) imports `from src.utils.gemini_client import GeminiClient` and uses it for batch explanation generation.